In [40]:
import pandas as pd
import numpy as np
from sklearn.metrics import mean_absolute_error

In [41]:
def load_mae(dataset: str, descriptor: str, target: str):
    datapath = f'../data/{dataset}/KRR_output'
    filename = f'mean_{descriptor}_{target}.csv'
    df = pd.read_csv(f'{datapath}/{filename}')
    maes = df.tail(1)['Test_MAE'].values[0]
    '''
    log_targets = ['log_p_sat', 'log_kwg', 'log_kwiomg']
    if target in log_targets:
        maes = np.power(10 * np.ones(maes.shape), maes)'
    '''
    return maes

def calculate_differences(descriptors: list[str], vals: list[float]):
    for idx, (descriptor, val) in enumerate(zip(descriptors, vals)):
        if idx == 0:
            last_desc = descriptor
            last_val = val
            continue
        else:
            print(f'{last_desc} - {descriptor}: {last_val - val:.4f}')
            last_desc = descriptor
            last_val = val

In [42]:
"""
def calc_mae_from_preds(dataset: str, descriptor: str, target: str,* , convert_log = False):
    random_state = [5, 12, 326, 432, 436, 452, 2435, 7543, 12343, 325432435]
    fpath = f'../data/{dataset}/KRR_output/output_predictions/'
    preds, reals = [], []
    #maes = []
    for seed in random_state:
        filename = f'output_predictions_{descriptor}_{target}_{seed}.csv'
        file = fpath + filename
        df = pd.read_csv(file, index_col='SMILES')
        pred, real = df['predictions'], df['target_values']
        if 'log' in target and convert_log:
            pred = np.power(10.0, pred)
            real = np.power(10.0, real)
        preds.append(pred)
        reals.append(real)
        #maes.append()
    preds, reals  = np.hstack(preds), np.hstack(reals)
    print(preds.shape, reals.shape)
    maes = mean_absolute_error(reals, preds)
    return maes
#calc_mae_from_preds('Wang', 'ATMOMACCS_DECIMAL_v4', 'log_p_sat', convert_log=True)
"""

"\ndef calc_mae_from_preds(dataset: str, descriptor: str, target: str,* , convert_log = False):\n    random_state = [5, 12, 326, 432, 436, 452, 2435, 7543, 12343, 325432435]\n    fpath = f'../data/{dataset}/KRR_output/output_predictions/'\n    preds, reals = [], []\n    #maes = []\n    for seed in random_state:\n        filename = f'output_predictions_{descriptor}_{target}_{seed}.csv'\n        file = fpath + filename\n        df = pd.read_csv(file, index_col='SMILES')\n        pred, real = df['predictions'], df['target_values']\n        if 'log' in target and convert_log:\n            pred = np.power(10.0, pred)\n            real = np.power(10.0, real)\n        preds.append(pred)\n        reals.append(real)\n        #maes.append()\n    preds, reals  = np.hstack(preds), np.hstack(reals)\n    print(preds.shape, reals.shape)\n    maes = mean_absolute_error(reals, preds)\n    return maes\n#calc_mae_from_preds('Wang', 'ATMOMACCS_DECIMAL_v4', 'log_p_sat', convert_log=True)\n"

In [43]:

def calc_mae_from_preds(dataset: str, descriptor: str, target: str,* , convert_log = False):
    random_state = [5, 12, 326, 432, 436, 452, 2435, 7543, 12343, 325432435]
    fpath = f'../data/{dataset}/KRR_output/output_predictions/'
    preds, reals = [], []
    #maes = []
    for seed in random_state:
        filename = f'output_predictions_{descriptor}_{target}_{seed}.csv'
        file = fpath + filename
        df = pd.read_csv(file, index_col='SMILES')
        pred, real = df['predictions'], df['target_values']
        #maes.append()
        preds.append(pred); reals.append(real)
    preds, reals  = np.hstack(preds), np.hstack(reals)
    #print(preds.shape, reals.shape)
    if 'log' in target and convert_log:
        maes = 10 ** mean_absolute_error(reals, preds)
    else:
        maes = mean_absolute_error(reals, preds)
    return maes
#calc_mae_from_preds('Wang', 'ATMOMACCS_DECIMAL_v4', 'log_p_sat', convert_log=True)

In [44]:
dataset = 'GeckoQ'
descriptor = 'ATMOMACCS_v1'
target = 'log_p_sat'
mae = load_mae(dataset, descriptor, target)
print(mae)

0.966176928980608


In [45]:
datasets = ['Wang', 'Ferraz-Caetano', 'Li', 'GeckoQ']
descriptors = ['ATMOMACCS_v1', 'ATMOMACCS_v2', 'ATMOMACCS_v3', 'ATMOMACCS_v4',
               'ATMOMACCS_DECIMAL_v4']
datasets_to_targets = {
    'Wang' : ['log_p_sat', 'log_kwg', 'log_kwiomg'],
    'Ferraz-Caetano' : ['dvap'],
    'Li': ['tg'],
    'GeckoQ': ['log_p_sat']
}
dataset_target_pairs = []
for dataset in datasets:
    targets = datasets_to_targets[dataset]
    for target in targets:
        dataset_target_pairs.append(f'{dataset}_{target}')

In [51]:
descriptors_all = descriptors + ['MACCS', 'ATMO_v5', 'ATMOMACCS_ALT_v3', 'TopFP']
results_grid = pd.DataFrame(np.zeros((len(descriptors_all), len(dataset_target_pairs))))
results_grid.index, results_grid.columns = descriptors_all, dataset_target_pairs
convert_log = False
for idx, dataset in enumerate(datasets):
    targets = datasets_to_targets[dataset]
    for target in targets:
        print(f'{dataset}, {target}')
        vals = []
        for descriptor in descriptors:
            vals.append(calc_mae_from_preds(dataset, descriptor, target, convert_log=convert_log))
        calculate_differences(descriptors, vals)
        descriptors_others = ['MACCS', 'ATMO_DECIMAL_v4', 'ATMOMACCS_ALT_v3', f'TopFP_{target}']
        for descriptor in descriptors_others:
            vals.append(calc_mae_from_preds(dataset, descriptor, target, convert_log=convert_log))
        results_grid[f'{dataset}_{target}'] = vals
        atmomaccs_alt3 = results_grid.loc['ATMOMACCS_ALT_v3'][f'{dataset}_{target}']
        atmomaccs2 = results_grid.loc['ATMOMACCS_v2'][f'{dataset}_{target}']
        print(f'ATMOMACCS_v2 - ATMOMACCS_ALT_v3: {atmomaccs2 - atmomaccs_alt3:.4f}')
        print('-'*20)

Wang, log_p_sat
ATMOMACCS_v1 - ATMOMACCS_v2: 0.0532
ATMOMACCS_v2 - ATMOMACCS_v3: 0.0409
ATMOMACCS_v3 - ATMOMACCS_v4: 0.0025
ATMOMACCS_v4 - ATMOMACCS_DECIMAL_v4: 0.0110
ATMOMACCS_v2 - ATMOMACCS_ALT_v3: -0.0015
--------------------
Wang, log_kwg
ATMOMACCS_v1 - ATMOMACCS_v2: 0.0344
ATMOMACCS_v2 - ATMOMACCS_v3: 0.0030
ATMOMACCS_v3 - ATMOMACCS_v4: 0.0022
ATMOMACCS_v4 - ATMOMACCS_DECIMAL_v4: 0.0355
ATMOMACCS_v2 - ATMOMACCS_ALT_v3: 0.0033
--------------------
Wang, log_kwiomg
ATMOMACCS_v1 - ATMOMACCS_v2: 0.0553
ATMOMACCS_v2 - ATMOMACCS_v3: 0.0413
ATMOMACCS_v3 - ATMOMACCS_v4: 0.0002
ATMOMACCS_v4 - ATMOMACCS_DECIMAL_v4: 0.0133
ATMOMACCS_v2 - ATMOMACCS_ALT_v3: 0.0006
--------------------
Ferraz-Caetano, dvap
ATMOMACCS_v1 - ATMOMACCS_v2: 0.0871
ATMOMACCS_v2 - ATMOMACCS_v3: 7.2111
ATMOMACCS_v3 - ATMOMACCS_v4: -0.0167
ATMOMACCS_v4 - ATMOMACCS_DECIMAL_v4: 0.3957
ATMOMACCS_v2 - ATMOMACCS_ALT_v3: -0.0588
--------------------
Li, tg
ATMOMACCS_v1 - ATMOMACCS_v2: 1.4282
ATMOMACCS_v2 - ATMOMACCS_v3: 2.735

In [47]:
results_grid

,Wang_log_p_sat,Wang_log_kwg,Wang_log_kwiomg,Ferraz-Caetano_dvap,Li_tg,GeckoQ_log_p_sat
ATMOMACCS_v1,0.389825,0.463268,0.369449,10.103286,21.983811,0.966177
ATMOMACCS_v2,0.336627,0.428896,0.314195,10.016179,20.555653,0.781868
ATMOMACCS_v3,0.295720,0.425863,0.272944,2.805060,17.819845,0.730383
ATMOMACCS_v4,0.293213,0.423623,0.272761,2.821797,17.241462,0.728304
ATMOMACCS_DECIMAL_v4,0.282179,0.388080,0.259479,2.426138,18.307454,0.697484
MACCS,0.435449,0.520869,0.413193,10.095667,22.025290,1.111058
ATMO_v5,0.428482,0.609008,0.383575,5.032780,22.175237,0.823694
ATMOMACCS_ALT_v3,0.338087,0.425628,0.313567,10.074945,19.762399,0.785018
TopFP,0.305208,0.409991,0.284738,6.288985,23.458326,0.753932


In [48]:
topfp = results_grid.loc['TopFP'].values
atmomaccs = results_grid.loc['ATMOMACCS_DECIMAL_v4'].values
1-max(atmomaccs / topfp)

0.053442349952945056

In [49]:
reductions = 1 - atmomaccs / topfp
diffs = topfp - atmomaccs
labels = results_grid.columns
units = {
    'Wang_log_p_sat' : 'log10(kPa)',
    'GeckoQ_log_p_sat' : 'log10(kPa)',
    'Wang_log_kwg' : 'log(1)',
    'Wang_log_kwiomg' : 'log(1)',
    'Ferraz-Caetano_dvap' : 'kJ / mol',
    'Li_tg' : 'K'
}
print('ATMOMACCS is better than TopFP by:')
for label, reduction, diff in zip(labels, reductions, diffs):
    if label == 'Li_tg':
        reduction = 1 - results_grid['Li_tg'].loc['ATMOMACCS_v4'] / \
                    results_grid['Li_tg'].loc['TopFP']
        diff = results_grid['Li_tg'].loc['TopFP'] - results_grid['Li_tg'].loc['ATMOMACCS_v4']
    unit = units[label]
    print('-' * 40)
    print(f'|    {label} : {100 * reduction:.3f} %, \n|    {label} : {diff:.3f} ({unit})')
print('-' * 40)

ATMOMACCS is better than TopFP by:
----------------------------------------
|    Wang_log_p_sat : 7.546 %, 
|    Wang_log_p_sat : 0.023 (log10(kPa))
----------------------------------------
|    Wang_log_kwg : 5.344 %, 
|    Wang_log_kwg : 0.022 (log(1))
----------------------------------------
|    Wang_log_kwiomg : 8.871 %, 
|    Wang_log_kwiomg : 0.025 (log(1))
----------------------------------------
|    Ferraz-Caetano_dvap : 61.422 %, 
|    Ferraz-Caetano_dvap : 3.863 (kJ / mol)
----------------------------------------
|    Li_tg : 26.502 %, 
|    Li_tg : 6.217 (K)
----------------------------------------
|    GeckoQ_log_p_sat : 7.487 %, 
|    GeckoQ_log_p_sat : 0.056 (log10(kPa))
----------------------------------------


In [50]:
'''
dataset_to_descriptors = {
    'Wang': ['MACCS', 'TopFP_log_p_sat', 'TopFP_log_kwg', 'TopFP_log_kwiomg'],
    'GeckoQ' : ['MACCS', 'TopFP_log_p_sat'],
    'Li' : ['MACCS', 'TopFP_tg'],
    'Ferraz-Caetano' : ['MACCS', 'TopFP_dvap']
}
for idx, dataset in enumerate(datasets):
    targets = datasets_to_targets[dataset]
    for target in targets:
        print(f'{dataset}, {target}')
        vals = []
        descriptors = ['MACCS', 'ATMO_DECIMAL_v4']
        descriptors.append(f'TopFP_{target}')
        for descriptor in descriptors:
            vals.append(load_mae(dataset, descriptor, target))
        results_grid[f'{dataset}_{target}'] = vals
        print('-'*20)'
'''

"\ndataset_to_descriptors = {\n    'Wang': ['MACCS', 'TopFP_log_p_sat', 'TopFP_log_kwg', 'TopFP_log_kwiomg'],\n    'GeckoQ' : ['MACCS', 'TopFP_log_p_sat'],\n    'Li' : ['MACCS', 'TopFP_tg'],\n    'Ferraz-Caetano' : ['MACCS', 'TopFP_dvap']\n}\nfor idx, dataset in enumerate(datasets):\n    targets = datasets_to_targets[dataset]\n    for target in targets:\n        print(f'{dataset}, {target}')\n        vals = []\n        descriptors = ['MACCS', 'ATMO_DECIMAL_v4']\n        descriptors.append(f'TopFP_{target}')\n        for descriptor in descriptors:\n            vals.append(load_mae(dataset, descriptor, target))\n        results_grid[f'{dataset}_{target}'] = vals\n        print('-'*20)'\n"